# Deterministic Opportunity Screening (Feature 7, Slices 7A–7C)

Every layer below this one answers a question about a *company*. This is the first layer that
answers a question about *research*:

> Which companies look economically strong enough, improving enough, and resilient enough to
> deserve expensive underwriting?

It deliberately does **not** answer:

> Is this stock cheap, and should I buy it?

Valuation comes next. Feature 7 contains no price, multiple, intrinsic value, expected return,
scenario, probability, or position size — and no LLM. Every band, gate, bucket, and reason is a
documented deterministic rule over facts OwnerLens already derived:

```
owner economics ─┐
capital efficiency ├─→ six dimension bands ─→ gates ─→ bucket ─→ reasons
economic value     │
compounding        │
capital allocation ┘        ↑
economic summary       Feature 6 coverage sets the ceiling
```

Run the screen yourself with `uv run python scripts/screen_universe.py`, or
`uv run owner-lens screen` over whatever you have persisted.

In [ ]:
import json
from pathlib import Path

from owner_lens.persistence import FilesystemRawSnapshotStore
from owner_lens.screening import (
    DIMENSION_ORDER,
    CoverageClass,
    ScreeningBucket,
    ScreeningDimension,
    assemble_screening_evidence,
    format_screening_result,
    format_screening_table,
    screen_universe,
)
from owner_lens.sec_adapter import canonical_history_from_sec

repo_root = next((p for p in (Path('.'), Path('..')) if (p / 'pyproject.toml').exists()), Path('.'))
manifest = json.loads((repo_root / 'data' / 'universe_6a_report.json').read_text(encoding='utf-8'))
raw_store = FilesystemRawSnapshotStore(repo_root / 'data' / 'raw' / 'sec' / 'company_facts')

histories = [
    canonical_history_from_sec(
        json.loads(raw_store.get(entry['content_hash'])), ticker=entry['ticker'], max_years=5
    )
    for entry in manifest['companies']
    if entry.get('content_hash') and raw_store.exists(entry['content_hash'])
]
screening = screen_universe(histories)
by_ticker = {result.ticker: result for result in screening.results}
print(f'{len(histories)} companies screened, no network call')

## 1. What the data forced

The design is not a set of ratios chosen in advance. Three measured facts shaped every rule.

**Coverage is the binding constraint.** Only 7 of 24 companies have every layer available. Eight of
the ten `PARTIAL` companies have *only* owner economics — no ROIC, no balance sheet, no capital
allocation. A screener that consumed the Feature 2 company-level summary alone would return
"insufficient data" for 17 of 24 companies, so screening composes from the **component** layers and
records exactly which ones were obtainable.

In [ ]:
coverage_counts = {}
for result in screening.results:
    coverage_counts[result.coverage_class.value] = coverage_counts.get(result.coverage_class.value, 0) + 1

print('Coverage classes:', coverage_counts)
print()
print(f'{"TKR":<6}{"COVERAGE":<10}{"EVALUABLE":>10}   unavailable dimensions')
for result in sorted(screening.results, key=lambda r: (r.coverage_class.value, r.ticker)):
    missing = ', '.join(d.value.lower() for d in result.unavailable_dimensions)
    print(f'{result.ticker:<6}{result.coverage_class.value:<10}{result.evaluable_dimensions:>10}   {missing}')

**ROIC level and ROIC trend must be separated.** Microsoft's ROIC fell 44 percentage points over
the window, from roughly 80% to 36%. That is not a collapse in earning power: invested capital is
debt plus equity minus cash, so spending down a vast net-cash pile mechanically enlarges the
denominator. A 36% return is still exceptional.

So the **level** sets the capital-efficiency band, and the **fall** is reported as a reason and
scored under economic momentum. The guardrail demotes only when the resulting level is *also* below
the strong threshold.

In [ ]:
msft = assemble_screening_evidence(next(h for h in histories if h.ticker == 'MSFT'))
print('MSFT ROIC by year:', [f'{row.roic:.1%}' for row in msft.capital_efficiency if row.roic])
print('MSFT net cash by year ($B):', [round(row.net_cash / 1e9, 1) for row in msft.capital_efficiency if row.net_cash])
print()
score = by_ticker['MSFT'].dimensions[ScreeningDimension.CAPITAL_EFFICIENCY]
print('Capital efficiency band:', score.band.value)
print('Reasons:', [r.value for r in score.reasons])

**Margin levels must never be scored across companies.** Costco runs a 3.8% operating margin and a
2.8% FCF margin against a 40.9% ROIC. Any absolute margin band makes Costco look weak and bakes
business-model bias into the screen permanently.

So business quality reads *cash-generation durability* for its base band and *margin trend* for a
bounded one-notch modifier. Margin levels appear in the evidence and never in the band. The only
absolute cross-company level threshold in the whole design is ROIC, which is already
capital-turnover adjusted — which is exactly why a low-margin, high-turnover retailer scores
correctly on it.

In [ ]:
for ticker in ('COST', 'ADBE'):
    evidence = assemble_screening_evidence(next(h for h in histories if h.ticker == ticker))
    latest = evidence.owner_economics[-1]
    quality = by_ticker[ticker].dimensions[ScreeningDimension.BUSINESS_QUALITY]
    efficiency = by_ticker[ticker].dimensions[ScreeningDimension.CAPITAL_EFFICIENCY]
    print(
        f'{ticker}: operating margin {latest.operating_margin:.1%}, FCF margin {latest.fcf_margin:.1%}'
        f'  ->  quality {quality.band.value}, capital efficiency {efficiency.band.value}'
    )

## 2. Six dimensions, coarse bands

Each dimension resolves to `STRONG`, `ADEQUATE`, `WEAK`, `POOR`, or `NOT_EVALUABLE`. No decimals, no
false precision: a 13% FCF/share CAGR and an 87% one both land in `STRONG`, which is precisely what
stops one extraordinary number from deciding a ranking.

| Dimension | Question | Needs |
|---|---|---|
| Business quality | Does it reliably convert activity into cash? | owner economics only |
| Capital efficiency | What return does it earn on capital? | capital efficiency |
| Per-share compounding | Is value per ownership unit growing? | diluted shares |
| Balance-sheet strength | Can it survive a bad outcome unforced? | capital efficiency |
| Capital allocation | Does management convert cash into owner value? | capital allocation |
| Economic momentum | Are economics improving or deteriorating? | economic value (with two fallbacks) |

`NOT_EVALUABLE` is never a zero and never a neutral score. It is excluded from the score and from
the evaluable count, and it triggers a coverage ceiling.

In [ ]:
abbr = {'STRONG': 'STR', 'ADEQUATE': 'ADQ', 'WEAK': 'WEK', 'POOR': 'POR', 'NOT_EVALUABLE': '---'}
print(f'{"TKR":<6}' + ''.join(f'{d.value[:4]:<6}' for d in DIMENSION_ORDER))
for result in screening.ranked:
    if result.coverage_class is CoverageClass.FAILED:
        continue
    print(f'{result.ticker:<6}' + ''.join(f'{abbr[result.band(d).value]:<6}' for d in DIMENSION_ORDER))

## 3. Gate, score, or both?

**Pure weighted scoring fails on this data.** With eight companies evaluable on two of six
dimensions, a weighted average either silently renormalizes — which rewards *absent* negative
evidence — or imputes neutral values, which fabricates facts the constitution forbids. It also
cannot express "two straight years of cash burn disqualifies regardless of everything else".

**Pure gating fails too.** Gates are binary. They cannot rank the seven fully covered companies, and
they cannot express "intact foundation, temporarily depressed economics".

**So: hybrid.** Gates are survival and usability filters only — never quality preferences, which is
what "leverage last" means in practice. Everything that survives is banded and bucketed by an
explicit rule table.

| Gate | Fires when | Result |
|---|---|---|
| G0a | coverage `FAILED` | `INSUFFICIENT_DATA` |
| G0b | fewer than 2 evaluable dimensions, or no business quality | `INSUFFICIENT_DATA` |
| G1 | free cash flow ≤ 0 in both of the last two years | `LOW_PRIORITY` |
| G2 | net debt above 6× free cash flow, or net debt with no free cash flow | `LOW_PRIORITY` |
| G3 | ROIC below 5% every year and not materially improving | `LOW_PRIORITY` |
| G4 | diluted-share CAGR at or above +5%/yr | `LOW_PRIORITY` |

G2 and G3 are **skipped**, never assumed, when the balance sheet is invisible. A gated company
still reports every dimension it could measure, so the reader always sees why it was set aside.

In [ ]:
gated = [r for r in screening.results if r.gate and r.coverage_class is not CoverageClass.FAILED]
for result in gated:
    print(f'{result.ticker}: {result.gate.value}')
    evidence = assemble_screening_evidence(next(h for h in histories if h.ticker == result.ticker))
    print('   FCF by year ($M):', [None if r.free_cash_flow is None else round(r.free_cash_flow / 1e6) for r in evidence.owner_economics])
    print('   dimensions still reported:', {d.value[:4]: result.band(d).value for d in DIMENSION_ORDER})

## 4. Buckets are research priority, not advice

The bucket comes from a rule table over named dimensions, so it can always be explained by pointing
at the rule that matched. The integer `screening_score` exists only to order companies *within* a
bucket; it never sets one, and it is not comparable across coverage classes.

* **`HIGH_PRIORITY`** — full coverage, strong quality, compounding and momentum at least adequate,
  nothing below adequate, and at least four of six strong. The "four of six" requirement is what
  keeps a single threshold boundary from deciding the top bucket on its own.
* **`WORTH_UNDERWRITING`** — strong with one real caveat; *or* the asymmetric shape (intact quality,
  returns, and balance sheet with depressed compounding or momentum); *or* coverage-limited with
  everything visible at least adequate and two strong.
* **`WATCH`** — at least two evaluable dimensions adequate or better, at most one poor.
* **`LOW_PRIORITY`** — anything else, or gated.
* **`INSUFFICIENT_DATA`** — cannot be screened at all.

In [ ]:
print(format_screening_table(screening))

## 5. `PARTIAL` coverage gets a ceiling, not a discount

A partially covered company is missing evidence in **both** directions: its unseen dimensions could
be strengths or weaknesses. Scoring it on what is visible and ranking it beside a fully measured
company would let absent negative evidence become an advantage.

So a `PARTIAL` company is capped:

* three or more evaluable dimensions → ceiling `WORTH_UNDERWRITING`
* fewer than three → ceiling `WATCH`

ServiceNow is the test of whether this is honest. Its measurable economics are as good as the
leaders': 25% FCF/share CAGR, expanding margins, barely any dilution. Its balance sheet is simply
unknowable from SEC Company Facts today. It reaches `WORTH_UNDERWRITING` — high enough that a human
will look — and no higher.

In [ ]:
now = by_ticker['NOW']
adbe = by_ticker['ADBE']
print(f'NOW   bucket {now.bucket.value:<20} ceiling {now.coverage_ceiling.value:<20} score {now.screening_score}')
print(f'ADBE  bucket {adbe.bucket.value:<20} ceiling {adbe.coverage_ceiling!s:<20} score {adbe.screening_score}')
print()
print('NOW outranks ADBE?', now.rank_key < adbe.rank_key)
print('NOW blocking metrics:', now.unavailable_metrics)

### Weak fundamentals versus incomplete data

The screen must never confuse "we measured this and it is weak" with "we could not measure this".
Every result carries a `limiting_factor` of `FUNDAMENTALS`, `COVERAGE`, `BOTH`, or `NONE`.

This also quantifies what the Feature 6 coverage gap costs: Home Depot, Lowe's, Lululemon, and
Coca-Cola are judged on owner economics alone, and their results say so.

In [ ]:
print(f'{"TKR":<6}{"BUCKET":<20}{"LIMITED BY":<14}gate')
for result in screening.ranked:
    print(f'{result.ticker:<6}{result.bucket.value:<20}{result.limiting_factor.value:<14}{result.gate.value if result.gate else ""}')

## 6. Compounder or asymmetric setup?

Priority says *when* to look. Setup type says *why*, and the two are deliberately orthogonal.

* `COMPOUNDER` — already excellent and still compounding.
* `POTENTIAL_ASYMMETRIC_SETUP` — returns on capital and balance sheet intact, recent economics
  depressed. This is the shape a temporarily impaired quality business takes.
* `NEITHER` — including every company whose returns on capital cannot be seen, because neither label
  can be granted on cash-flow evidence alone.

Microsoft is the live example, and it is the clearest demonstration that the screen is not a
momentum filter: Feature 2 calls it `STRONGLY_DETERIORATING`, yet its ROIC is 36% and it holds $36B
of net cash. It is set aside from the top bucket and flagged for underwriting, not discarded.

In [ ]:
print(format_screening_result(by_ticker['MSFT']))

Compare against a company whose economics are deteriorating *and* whose foundation is not intact.
The asymmetric path requires strong quality, strong returns, and an adequate balance sheet, so it
is not a rule for catching falling knives.

In [ ]:
print(format_screening_result(by_ticker['CRM']))

## 7. Does it avoid the obvious failure modes?

Four checks, each a test in `tests/test_screening_universe.py`.

In [ ]:
nvda, adbe, cmg, cost = (by_ticker[t] for t in ('NVDA', 'ADBE', 'CMG', 'COST'))

print('1. NVDA does not dominate on one extraordinary metric')
print(f'   NVDA ROIC 94.5%, FCF/share CAGR 87% -> score {nvda.screening_score}, {nvda.bucket.value}')
print(f'   ADBE ROIC 61.6%, FCF/share CAGR 13% -> score {adbe.screening_score}, {adbe.bucket.value}')
print()
print('2. COST is not penalized for a 3.8% operating margin')
print(f'   quality {cost.band(ScreeningDimension.BUSINESS_QUALITY).value}, '
      f'capital efficiency {cost.band(ScreeningDimension.CAPITAL_EFFICIENCY).value}, {cost.bucket.value}')
print()
print('3. Within every bucket, full coverage ranks ahead of partial coverage')
for bucket in ScreeningBucket:
    members = screening.in_bucket(bucket)
    if any(r.coverage_class is CoverageClass.PARTIAL for r in members) and any(
        r.coverage_class is CoverageClass.FULL for r in members
    ):
        order = ', '.join(f'{r.ticker} ({r.coverage_class.value})' for r in members)
        print(f'   {bucket.value}: {order}')
print()
print('4. The universe actually narrows')
worth = screening.worth_underwriting()
print(f'   {len(worth)} of {len(screening.results)} warrant deeper work: {", ".join(r.ticker for r in worth)}')

## 8. Where this stops

Feature 7 ends here, on purpose. It has produced a short list and an auditable reason for every
name on and off it. It has said nothing about what any of them is worth.

What comes next — intrinsic value, bear/base/bull distributions, expected return, edge, position
sizing — all needs a price, and a price is the first input in this entire system that OwnerLens does
not derive from a filing. Keeping it out of this layer is what lets the short list be re-derived
identically tomorrow, and what makes the valuation layer's assumptions visible when it arrives.

The other finding worth carrying forward: **four companies rank low partly because only owner
economics is visible for them.** That is the sharpest measurement yet of what the Feature 6 coverage
gap costs, and the `limiting_factor` field is what will tell us whether closing it is worth doing.